# Designing Tool Schemas LLMs Can Reliably Use

Tool calling fails more often from **bad schemas** than from weak models. A schema is the **contract** between your agent runtime and the LLM: parameter names, types, descriptions, and constraints determine whether the model picks the right tool with valid arguments.

The model never sees your Python function. It only sees:

- the tool **name**,
- a natural-language **description** (when to use it),
- a **parameters** object (JSON Schema) describing each argument.

This notebook teaches how to design schemas that models invoke reliably. We use **Pydantic** to define argument models, generate JSON Schema, validate inputs before execution, and compare **bad vs good** designs side by side. The scenario is a fictional **SaaS Subscription & Billing API** — plans, subscriptions, invoices, and support tickets.

Everything is self-contained. No prior notebooks or orchestration frameworks are required.

In [ ]:
"""
SaaS Subscription & Billing API — backend for schema design demos.
"""

import json
import os
import re
import uuid
from enum import Enum
from typing import Any, Callable, Literal

from pydantic import BaseModel, Field, ValidationError, field_validator

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


# --- In-memory SaaS backend ---
PLANS: dict[str, dict[str, Any]] = {
    "starter": {"name": "Starter", "price_monthly": 19.0, "seats": 3, "api_calls": 10_000},
    "pro": {"name": "Pro", "price_monthly": 49.0, "seats": 15, "api_calls": 100_000},
    "enterprise": {"name": "Enterprise", "price_monthly": 199.0, "seats": 100, "api_calls": 1_000_000},
}

SUBSCRIPTIONS: dict[str, dict[str, Any]] = {
    "SUB-1001": {
        "customer_email": "alice@acme.com",
        "plan_id": "pro",
        "status": "active",
        "billing_email": "billing@acme.com",
        "renewal_date": "2026-08-01",
    },
}

BILLING_DOCS: list[dict[str, str]] = [
    {"id": "doc-refund", "text": "Refunds available within 14 days of annual plan purchase. Monthly plans are non-refundable."},
    {"id": "doc-invoice", "text": "Invoices are emailed on the 1st of each month. Download PDFs from Billing > Invoices."},
    {"id": "doc-seats", "text": "Adding seats mid-cycle is prorated. Removing seats takes effect at next renewal."},
    {"id": "doc-tax", "text": "VAT applied for EU customers. Tax ID can be added in Billing > Tax settings."},
]

print(f"Plans: {list(PLANS.keys())} | Subscriptions: {len(SUBSCRIPTIONS)} | Docs: {len(BILLING_DOCS)}")

---

## 1. Schema as the Agent–Computer Interface

The schema is how the model understands what it **can** do and **how** to do it:

```
User: "How do I get a refund on my Pro plan?"
        │
        ▼
┌──────────────────────────────────────────────┐
│  Tool menu (schemas sent to the model)        │
│  • search_billing_docs(query: str)          │
│  • get_subscription(subscription_id: enum)  │
│  • list_plans()                             │
└──────────────────────────────────────────────┘
        │
        ▼
tool_call: { name: "search_billing_docs", arguments: {"query": "refund pro"} }
```

| Schema field | Model uses it to… |
|--------------|-------------------|
| **`name`** | Choose among tools |
| **`description`** | Decide *when* to call (vs sibling tools) |
| **`parameters.properties`** | Know which argument keys exist |
| **`enum` / `required`** | Constrain valid values |

---

## 2. Pydantic Models → JSON Schema

**Pydantic** `BaseModel` classes are the standard way to define tool argument schemas in Python. Calling `model_json_schema()` produces JSON Schema that OpenAI, Anthropic, and other providers accept.

The most important reliability lever is **`Field(description=...)`** on every parameter — the model reads these descriptions when filling arguments.

In [ ]:
class PlanId(str, Enum):
    """Closed set of subscription plan identifiers."""
    starter = "starter"
    pro = "pro"
    enterprise = "enterprise"


class SubscriptionId(str, Enum):
    """Known subscription IDs in the billing system."""
    sub_1001 = "SUB-1001"


class GetSubscriptionInput(BaseModel):
    subscription_id: SubscriptionId = Field(
        description="Subscription ID exactly as shown in Billing dashboard (e.g. SUB-1001)."
    )


class SearchBillingDocsInput(BaseModel):
    query: str = Field(
        min_length=2,
        max_length=120,
        description=(
            "Short keyword query for billing documentation — e.g. 'refund', "
            "'invoice download', 'add seats', 'VAT tax'."
        ),
    )
    max_results: int = Field(
        default=3,
        ge=1,
        le=5,
        description="Maximum number of documentation snippets to return (1-5).",
    )


print("GetSubscriptionInput JSON Schema:")
print(pretty(GetSubscriptionInput.model_json_schema()))
print("\nSearchBillingDocsInput JSON Schema (excerpt):")
schema = SearchBillingDocsInput.model_json_schema()
print(pretty(schema["properties"]))

---

## 3. Converting Pydantic to OpenAI Tool Format

Providers expect a specific wrapper shape. This helper converts any Pydantic model into the `tools` array entry the API accepts.

In [ ]:
def pydantic_to_openai_tool(
    name: str,
    description: str,
    args_model: type[BaseModel],
) -> dict[str, Any]:
    """Wrap a Pydantic model as an OpenAI-compatible tool schema."""
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": args_model.model_json_schema(),
        },
    }


OPENAI_TOOLS = [
    pydantic_to_openai_tool(
        name="get_subscription",
        description=(
            "Fetch details for a specific subscription when you already know the subscription ID. "
            "Do NOT use when the user only describes their plan name — use list_plans or search_billing_docs."
        ),
        args_model=GetSubscriptionInput,
    ),
    pydantic_to_openai_tool(
        name="search_billing_docs",
        description=(
            "Search billing policy documentation when the user asks how something works "
            "(refunds, invoices, seats, tax). Do NOT use when you have a subscription ID — use get_subscription."
        ),
        args_model=SearchBillingDocsInput,
    ),
]

print(f"Built {len(OPENAI_TOOLS)} OpenAI tool schemas")
print("\nExample — get_subscription:")
print(pretty(OPENAI_TOOLS[0])[:600], "...")

---

## 4. Bad vs Good — `get_subscription`

| Anti-pattern | Why it hurts |
|--------------|--------------|
| Parameter named `id` or `x` | Ambiguous — user id? plan id? invoice id? |
| Type `str` with no enum | Model invents `sub_1`, `SUB1001`, `alice` |
| Description: "Get subscription" | No cue *when* vs `search_billing_docs` |

Compare the JSON Schema the model actually receives:

In [ ]:
class BadGetSubscription(BaseModel):
    id: str = Field(description="Get subscription")


class GoodGetSubscription(BaseModel):
    subscription_id: SubscriptionId = Field(
        description=(
            "Subscription ID from Billing dashboard (SUB-1001). "
            "Use only when the customer provides this exact ID."
        ),
    )


print("BAD schema — free-form id:")
print(pretty(BadGetSubscription.model_json_schema()["properties"]))

print("\nGOOD schema — enum constraint:")
print(pretty(GoodGetSubscription.model_json_schema()["properties"]))

---

## 5. Bad vs Good — Avoiding Tool Overload

**Tool overload** happens when one schema tries to handle multiple modes via an `action` parameter or vague flags. Models confuse modes and hallucinate extra keys.

```
BAD:  manage_billing(action=search|get|update|cancel, target=..., ...)
GOOD: search_billing_docs(...), get_subscription(...), update_billing_email(...)
```

Split overloaded tools into **focused** functions with **non-overlapping** descriptions.

In [ ]:
class BadBillingTool(BaseModel):
    action: str = Field(description="search or get or update or cancel")
    target: str = Field(description="what to look for or change")
    value: str = Field(default="", description="optional value")


class GoodSearchBillingDocs(BaseModel):
    query: str = Field(description="Keywords to match billing documentation.")
    max_results: int = Field(default=3, ge=1, le=5)


class GoodUpdateBillingEmail(BaseModel):
    subscription_id: SubscriptionId = Field(description="Subscription to update.")
    new_email: str = Field(description="New billing contact email (must contain @).")

    @field_validator("new_email")
    @classmethod
    def validate_email(cls, v: str) -> str:
        if "@" not in v:
            raise ValueError("must be a valid email address")
        return v.lower().strip()


print("BAD — mode switch confuses the model:")
print(f"  properties: {list(BadBillingTool.model_json_schema()['properties'].keys())}")

print("\nGOOD — separate tools, separate jobs:")
print(f"  search properties: {list(GoodSearchBillingDocs.model_json_schema()['properties'].keys())}")
print(f"  update properties: {list(GoodUpdateBillingEmail.model_json_schema()['properties'].keys())}")

---

## 6. Implementing Tools Behind the Schemas

Schemas describe the interface; Python functions implement the behavior. Always **validate** with Pydantic before executing.

In [ ]:
def search_billing_docs(query: str, max_results: int = 3) -> list[dict[str, str]]:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    hits = []
    for doc in BILLING_DOCS:
        if any(t in doc["text"].lower() for t in terms):
            hits.append(doc)
        if len(hits) >= max_results:
            break
    return hits or BILLING_DOCS[:1]


def get_subscription(subscription_id: str) -> dict[str, Any]:
    sub = SUBSCRIPTIONS.get(subscription_id.upper())
    if not sub:
        return {"found": False, "error": f"Subscription {subscription_id} not found"}
    plan = PLANS.get(sub["plan_id"], {})
    return {"found": True, "subscription_id": subscription_id.upper(), **sub, "plan": plan}


def list_plans() -> list[dict[str, Any]]:
    return [{"plan_id": k, **v} for k, v in PLANS.items()]


def update_billing_email(subscription_id: str, new_email: str) -> dict[str, Any]:
    sub = SUBSCRIPTIONS.get(subscription_id.upper())
    if not sub:
        return {"success": False, "error": "subscription not found"}
    sub["billing_email"] = new_email
    return {"success": True, "subscription_id": subscription_id.upper(), "billing_email": new_email}


TOOL_IMPLEMENTATIONS: dict[str, Callable[..., Any]] = {
    "search_billing_docs": search_billing_docs,
    "get_subscription": get_subscription,
    "list_plans": lambda: list_plans(),
    "update_billing_email": update_billing_email,
}

# Demo
print(search_billing_docs("refund annual"))
print(get_subscription("SUB-1001")["plan"]["name"])

---

## 7. Enum and Literal Parameters

Closed sets communicate valid values better than free-form strings.

| Approach | JSON Schema output | Best for |
|----------|-------------------|----------|
| **`Enum`** | `"enum": ["starter", "pro", ...]` | Stable identifiers |
| **`Literal["a","b"]`** | same | Small fixed sets |
| **`str` + regex** | `"pattern"` | Formats like email (prefer validator too) |

In [ ]:
class SearchByTopicInput(BaseModel):
    topic: Literal["refunds", "invoices", "seats", "tax"] = Field(
        description="High-level billing topic area. Use when user question maps to one category."
    )


TOPIC_TO_DOCS = {
    "refunds": ["doc-refund"],
    "invoices": ["doc-invoice"],
    "seats": ["doc-seats"],
    "tax": ["doc-tax"],
}


def search_by_topic(topic: Literal["refunds", "invoices", "seats", "tax"]) -> str:
    doc_ids = TOPIC_TO_DOCS[topic]
    lines = []
    for doc in BILLING_DOCS:
        if doc["id"] in doc_ids:
            lines.append(f"[{doc['id']}] {doc['text']}")
    return "\n".join(lines)


print(search_by_topic("refunds"))
print("\nLiteral enum in schema:")
print(pretty(SearchByTopicInput.model_json_schema()["properties"]["topic"]))

---

## 8. Naming Conventions

Consistent names reduce cross-tool confusion.

| Rule | Good | Bad |
|------|------|-----|
| **verb_noun** | `search_billing_docs`, `get_subscription` | `docs`, `billingTool` |
| **snake_case** | `update_billing_email` | `updateBillingEmail` |
| **domain prefix** | `billing_search_docs` | `search` (too generic) |
| **no abbreviations** | `get_subscription` | `get_sub`, `gs` |

Keep tool names **stable** — renaming breaks stored traces, eval harnesses, and fine-tuned routers.

In [ ]:
def audit_tool_name(name: str) -> tuple[bool, str]:
    if not re.match(r"^[a-z][a-z0-9]*(_[a-z0-9]+)+$", name):
        return False, "use snake_case verb_noun (e.g. get_subscription)"
    if len(name) < 6:
        return False, "name too short — add domain context"
    vague = {"do_stuff", "handle", "process", "manage", "run"}
    if name.split("_")[0] in vague:
        return False, f"vague verb '{name.split('_')[0]}'"
    return True, "OK"


CANDIDATE_NAMES = [
    "search_billing_docs",
    "get_subscription",
    "do_stuff",
    "manage",
    "updateBillingEmail",
]

for name in CANDIDATE_NAMES:
    ok, msg = audit_tool_name(name)
    print(f"{'✓' if ok else '✗'} {name:<25} {msg}")

---

## 9. Few Tools per Agent — The Router Pattern

Models make more tool-selection errors as the menu grows. **3–5 tools per agent** is a practical sweet spot. When you need more capabilities, use a **router** that dispatches to specialized sub-agents with smaller toolsets.

```
User query
    │
    ▼
intent router ── billing_read_agent  [search_billing_docs, get_subscription, list_plans]
              ├── billing_write_agent [update_billing_email]  ← fewer tools, stricter guardrails
              └── chat_agent          (no tools)
```

In [ ]:
AGENT_TOOLSETS: dict[str, list[str]] = {
    "billing_read": ["search_billing_docs", "get_subscription", "list_plans"],
    "billing_write": ["update_billing_email"],
    "chat_only": [],
}

BLOATED_MENU = [
    "search_billing_docs", "get_subscription", "list_plans", "update_billing_email",
    "create_invoice", "cancel_subscription", "add_seats", "remove_seats",
    "search_users", "send_email", "run_report",
]


def classify_intent(message: str) -> str:
    t = message.lower()
    if any(w in t for w in ("change email", "update email", "billing email")):
        return "billing_write"
    if any(w in t for w in ("refund", "invoice", "plan", "subscription", "sub-")):
        return "billing_read"
    return "chat_only"


def tools_for_user(message: str) -> list[str]:
    intent = classify_intent(message)
    return AGENT_TOOLSETS.get(intent, [])


for msg in [
    "How do refunds work on annual plans?",
    "Change billing email for SUB-1001 to finance@acme.com",
    "Hello, what can you help with?",
]:
    tools = tools_for_user(msg)
    print(f"'{msg[:45]}...' → {tools or ['no tools']}")

print(f"\nBloated single agent: {len(BLOATED_MENU)} tools — SPLIT RECOMMENDED")
print(f"Router sub-agents: max {max(len(v) for v in AGENT_TOOLSETS.values())} tools each")

---

## 10. Descriptions That Teach *When* to Call

Structure tool descriptions as **purpose + when + when-not + example**:

1. **Purpose** — what the tool returns
2. **When** — contrast with sibling tools
3. **Do NOT use when** — prevent wrong tool selection
4. **Example args** — realistic values

In [ ]:
DESCRIPTION_TEMPLATE = """{name}: {purpose}
Use when: {when}
Do NOT use when: {avoid}
Example args: {example}"""


GOOD_DESCRIPTIONS = {
    "search_billing_docs": DESCRIPTION_TEMPLATE.format(
        name="search_billing_docs",
        purpose="Returns matching billing policy documentation snippets.",
        when="the user asks how billing works and you do not have a subscription ID.",
        avoid="you already have SUB-xxxx — use get_subscription instead.",
        example='{"query": "refund annual plan", "max_results": 2}',
    ),
    "get_subscription": DESCRIPTION_TEMPLATE.format(
        name="get_subscription",
        purpose="Returns live subscription details (plan, status, renewal date).",
        when="the customer provides a subscription ID like SUB-1001.",
        avoid="the user asks a general policy question — use search_billing_docs.",
        example='{"subscription_id": "SUB-1001"}',
    ),
}

for name, desc in GOOD_DESCRIPTIONS.items():
    print(f"--- {name} ---")
    print(desc)
    print()

---

## 11. Validate Args Before Side Effects

JSON Schema is the first line of defense. **Pydantic validation** is the second — especially before writes like `update_billing_email`.

Return validation errors as tool observations so the model can retry with corrected arguments.

In [ ]:
def safe_invoke(tool_name: str, raw_args: dict[str, Any], schema: type[BaseModel]) -> str:
    """Validate with Pydantic, then execute. Return JSON string for tool message."""
    try:
        parsed = schema.model_validate(raw_args)
    except ValidationError as exc:
        return json.dumps({
            "error": "validation_failed",
            "details": exc.errors(),
            "hint": "Fix arguments and retry.",
        })

    fn = TOOL_IMPLEMENTATIONS.get(tool_name)
    if not fn:
        return json.dumps({"error": f"unknown tool: {tool_name}"})

    result = fn(**parsed.model_dump())
    return json.dumps(result, default=str)


print("Valid call:")
print(safe_invoke("get_subscription", {"subscription_id": "SUB-1001"}, GetSubscriptionInput))

print("\nInvalid enum:")
print(safe_invoke("get_subscription", {"subscription_id": "SUB-9999"}, GetSubscriptionInput))

print("\nInvalid email:")
print(safe_invoke(
    "update_billing_email",
    {"subscription_id": "SUB-1001", "new_email": "not-an-email"},
    GoodUpdateBillingEmail,
))

---

## 12. Golden-Set Tool Selection Eval

Before shipping, run 10+ representative questions and verify the model picks the **expected tool**. Below is an offline rule-based stand-in; swap for a live LLM eval in production.

In [ ]:
GOLDEN_SET = [
    {"question": "How do refunds work for annual plans?", "expected_tool": "search_billing_docs"},
    {"question": "What plan is on subscription SUB-1001?", "expected_tool": "get_subscription"},
    {"question": "Compare Starter vs Pro pricing", "expected_tool": "list_plans"},
    {"question": "Change billing email to finance@acme.com for SUB-1001", "expected_tool": "update_billing_email"},
    {"question": "Where do I download invoices?", "expected_tool": "search_billing_docs"},
]


def offline_tool_router(question: str) -> str:
    """Rule-based stand-in for model tool selection."""
    q = question.lower()
    if "change" in q or "update" in q and "email" in q:
        return "update_billing_email"
    if "sub-" in q or "subscription sub" in q:
        return "get_subscription"
    if "compare" in q and "plan" in q or "starter" in q and "pro" in q:
        return "list_plans"
    return "search_billing_docs"


def eval_tool_selection(cases: list[dict]) -> float:
    hits = sum(1 for c in cases if offline_tool_router(c["question"]) == c["expected_tool"])
    return hits / len(cases)


print(f"{'Question':<50} {'Expected':<22} {'Predicted':<22} {'OK'}")
print("-" * 100)
for case in GOLDEN_SET:
    predicted = offline_tool_router(case["question"])
    ok = predicted == case["expected_tool"]
    print(f"{case['question']:<50} {case['expected_tool']:<22} {predicted:<22} {'✓' if ok else '✗'}")

print(f"\nAccuracy: {eval_tool_selection(GOLDEN_SET):.0%}")

---

## 13. Schema Review Checklist

Before shipping tools to production agents:

| # | Check |
|---|-------|
| 1 | Each tool does **one job** — no action/mode switches |
| 2 | Parameter names are **self-explanatory** (`subscription_id`, not `id`) |
| 3 | **`Field(description=...)`** on every argument |
| 4 | Closed sets use **`Enum`** or **`Literal`** |
| 5 | **≤ 5 tools** per agent — or route first |
| 6 | Descriptions include **when NOT** to call |
| 7 | Run golden questions — inspect actual `tool_calls` |
| 8 | Validate with Pydantic **before** side effects |

In [ ]:
def review_schema(tool_name: str, args_model: type[BaseModel], description: str) -> dict[str, Any]:
    """Automated checks for common schema issues."""
    schema = args_model.model_json_schema()
    props = schema.get("properties", {})
    issues = []

    ok_name, name_msg = audit_tool_name(tool_name)
    if not ok_name:
        issues.append(name_msg)
    if len(description) < 40:
        issues.append("description too short — add when/when-not")
    if "do not" not in description.lower() and "not use" not in description.lower():
        issues.append("missing 'do not use when' guidance")
    for pname, pdef in props.items():
        if pname in ("id", "x", "data", "value"):
            issues.append(f"vague parameter name: {pname}")
        if "description" not in pdef:
            issues.append(f"missing description on: {pname}")

    return {"tool": tool_name, "passed": len(issues) == 0, "issues": issues}


reviews = [
    review_schema("get_subscription", GetSubscriptionInput, GOOD_DESCRIPTIONS["get_subscription"]),
    review_schema("search_billing_docs", SearchBillingDocsInput, GOOD_DESCRIPTIONS["search_billing_docs"]),
    review_schema("do_stuff", BadBillingTool, "Do stuff with billing"),
]

for r in reviews:
    status = "PASS" if r["passed"] else "FAIL"
    print(f"{status}: {r['tool']}")
    for issue in r["issues"]:
        print(f"  - {issue}")

---

## 14. Common Failure Modes

| Symptom | Likely schema issue | Fix |
|---------|---------------------|-----|
| Wrong tool chosen | Overlapping descriptions | Add "Do NOT use when…" |
| Invalid enum value | `str` instead of `Enum` | Constrain allowed ids |
| Missing required arg | Everything optional | Mark truly required fields |
| Extra hallucinated keys | Loose schema | Use Pydantic models (sets `additionalProperties`) |
| Never calls tools | Weak descriptions | Add when/examples |
| Tool called with empty args | Missing `required` array | Check generated JSON Schema |

---

## 15. Optional — Live Tool Selection Test

Set `USE_LIVE_LLM = True` with a valid API key to see which tool a real model selects.

In [ ]:
USE_LIVE_LLM = False

class ListPlansInput(BaseModel):
    """No parameters — list all available subscription plans."""
    pass


ALL_OPENAI_TOOLS = OPENAI_TOOLS + [
    pydantic_to_openai_tool(
        name="list_plans",
        description="List all subscription plans with pricing. Use when user asks to compare plans.",
        args_model=ListPlansInput,
    ),
]

if USE_LIVE_LLM:
    try:
        from openai import OpenAI
        client = OpenAI()
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": "How do refunds work for annual Pro plan purchases?",
            }],
            tools=OPENAI_TOOLS,
        )
        msg = response.choices[0].message
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"Tool: {tc.function.name} | Args: {tc.function.arguments}")
        else:
            print("Text:", msg.content)
    except Exception as exc:
        print("LLM call failed:", exc)
else:
    print("Offline: golden-set eval above validates tool routing logic.")
    print("Expected for refund question: search_billing_docs")
    print("Predicted:", offline_tool_router("How do refunds work for annual Pro plan purchases?"))

---

## 16. Check Your Understanding

1. What three things does the model see for each tool (and what does it *not* see)?
2. Why is `subscription_id: SubscriptionId` more reliable than `id: str`?
3. What is tool overload and how do you fix it?
4. Why put "Do NOT use when" in tool descriptions?
5. How many tools per agent is a practical maximum before routing?
6. What is the difference between schema validation and business validation?
7. What should a golden-set eval measure before you ship tools?

---

## 17. Summary

Tool schemas are **UX for the model**. Invest in descriptions, enums, and focused tool boundaries.

**Key takeaways:**

- The model sees **name + description + JSON Schema** — never your Python function.
- Use **Pydantic** `Field(description=...)` on every parameter; generate schema with `model_json_schema()`.
- Split **overloaded** tools (`action=search|get|update`) into focused functions.
- Use **`Enum`** and **`Literal`** for closed sets — prevents invented ids.
- Follow **verb_noun** snake_case naming; avoid vague names like `id`, `do_stuff`, `manage`.
- Keep **≤ 5 tools per agent**; use a **router** for larger capability sets.
- Descriptions must teach **when** and **when NOT** to call each tool.
- **Validate with Pydantic before side effects**; return errors as tool observations for retry.
- Run a **golden-set eval** on tool selection before production.

Well-designed schemas are the foundation of reliable tool calling. The runtime loop executes tools; schemas teach the model how to invoke them correctly.